# 02 — Contribution, league factors, replacement level (Phase 2)

Plan: `docs/superpowers/plans/2026-08-28-phase2-contribution.md`. Each step below runs the smallest
thing that produces a real output, the decision is read off it and recorded in the note cell, and
only then the code is ported to `scout.models`. Inputs are the Phase 1 tables (`scout.panel`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 80); pd.set_option("display.max_rows", 300)
from scout import config
from scout.data import understat
from scout.panel import market, player_match, stints, workrate
LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}

## Step 1 — Which per-90 quantities carry signal, per role, and at what minutes floor?

Candidates from `player_match` (npxG, xA, key passes, shots, xGChain, xGBuildup) plus goals and
assists as the raw baseline. Per player × league-season × role (a player's season in one role),
per 90. Criterion: year-to-year r ≥ 0.3 for the same player in the same role, at floors 300 / 600 /
900 / 1,200 minutes; the floor is the lowest one at which r stops rising materially, weighed against
the share of the backtest population (paid Big-5 departures with a panel row) it drops.

In [2]:
pm = player_match.build()
pm["competition_id"] = pm.league.map(LEAGUE_TO_COMP)
shots = understat.load("shots")
print("shots columns:", shots.columns.tolist())
# penalties: situation NA with xG 0.7612 (notebook 01, Part 4b); own goals are not the shooter's
is_pen = shots.situation.isna() & (shots.xg.round(4) == 0.7612)
pen_xg = shots[is_pen].groupby(["game_id", "player_id"]).xg.sum().rename("pen_xg")
pm = pm.merge(pen_xg, on=["game_id", "player_id"], how="left").fillna({"pen_xg": 0.0})
pm["npxg"] = pm.xg - pm.pen_xg
print(len(pm), "player-match rows | penalty xG subtracted from", int((pm.pen_xg > 0).sum()), "rows | own-goal rows in shots:", int((shots.result == "OwnGoal").sum()) if "result" in shots else "n/a")

QUANTITIES = ["npxg", "xa", "key_passes", "shots", "xg_chain", "xg_buildup", "goals", "assists"]
season_role = pm.dropna(subset=["role"]).groupby(["competition_id", "season", "player_id", "role"]).agg(minutes=("minutes", "sum"), **{q: (q, "sum") for q in QUANTITIES}).reset_index()
per90 = season_role.copy()
for q in QUANTITIES:
    per90[q] = per90[q] / per90.minutes * 90
print(len(per90), "player-season-roles |", per90.groupby("role").size().to_dict())

shots columns: ['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'date', 'shot_id', 'team_id', 'player_id', 'assist_player_id', 'assist_player', 'xg', 'location_x', 'location_y', 'minute', 'body_part', 'situation', 'result']
630291 player-match rows | penalty xG subtracted from 1092 rows | own-goal rows in shots: 0
42157 player-season-roles | {'CB': 6770, 'CM': 8472, 'FB': 7317, 'GK': 2376, 'ST': 6179, 'W': 11043}


In [3]:
FLOORS = [300, 600, 900, 1200]


def year_to_year(frame, floor, quantities):
    kept = frame[frame.minutes >= floor]
    nxt = kept.assign(season=kept.season - 1)
    pairs = kept.merge(nxt, on=["competition_id", "player_id", "role", "season"], suffixes=("", "_next"))
    return pd.Series({q: pairs[q].corr(pairs[f"{q}_next"]) for q in quantities}), len(pairs)


rows = []
for role in ["GK", "CB", "FB", "CM", "W", "ST"]:
    for floor in FLOORS:
        r, n = year_to_year(per90[per90.role == role], floor, QUANTITIES)
        rows.append(pd.concat([pd.Series({"role": role, "floor": floor, "pairs": n}), r.round(2)]))
stability = pd.DataFrame(rows).set_index(["role", "floor"])
print(stability.to_string())
print("\nquantities with r >= 0.3 at 900, per role:")
for role, group in stability.xs(900, level="floor").iterrows():
    print(f"  {role}: {[q for q in QUANTITIES if group[q] >= 0.3]}")

            pairs  npxg    xa  key_passes  shots  xg_chain  xg_buildup  goals  assists
role floor                                                                            
GK   300     1020  0.06  0.04        0.12   0.02      0.56        0.56  -0.00     0.01
     600      890  0.15  0.07        0.11   0.05      0.65        0.65  -0.00     0.04
     900      800  0.15  0.06        0.12   0.08      0.64        0.64  -0.00     0.07
     1200     733  0.16  0.07        0.16   0.13      0.64        0.64  -0.00     0.08
CB   300     2966  0.27  0.21        0.40   0.46      0.67        0.67   0.13     0.06
     600     2492  0.30  0.24        0.45   0.52      0.71        0.71   0.16     0.08
     900     2047  0.33  0.28        0.49   0.55      0.73        0.72   0.15     0.10
     1200    1675  0.38  0.26        0.44   0.57      0.75        0.74   0.15     0.11
FB   300     2686  0.45  0.48        0.61   0.61      0.62        0.61   0.24     0.23
     600     2121  0.52  0.53        0.65  

In [4]:
# The cost side of the floor: paid departures from Big-5 clubs (backtest population) kept at each floor
moves = market.build()
paid = moves[(moves.kind == "paid")].copy()
paid["transfer_season"] = ("20" + paid.transfer_season.str[:2]).astype(int)
paid["prev_season"] = paid.transfer_season - (~pd.to_datetime(paid.transfer_date).dt.month.isin([1, 2, 3])).astype(int)
big5_clubs = stints.tm.load_player_club_seasons(list(config.BIG5), list(config.SEASONS))[["club_id"]].drop_duplicates()
paid = paid[paid.from_club_id.isin(big5_clubs.club_id) & paid.transfer_season.between(2015, 2024)]
last = stints.build(list(config.BIG5) + list(config.FEEDERS), list(config.SEASONS))[["tm_player_id", "club_id", "season", "minutes"]]
present = paid.merge(last, left_on=["player_id", "from_club_id", "prev_season"], right_on=["tm_player_id", "club_id", "season"]).dropna(subset=["minutes"])
print(f"paid Big-5 departures 15/16 → 24/25 with a panel row at the selling club: {len(present)} (of {len(paid)})")
print("kept at each floor:", {f: f"{(present.minutes >= f).mean():.1%}" for f in FLOORS}, "| fee share kept:", {f: f"{present.transfer_fee[present.minutes >= f].sum() / present.transfer_fee.sum():.1%}" for f in FLOORS})

paid Big-5 departures 15/16 → 24/25 with a panel row at the selling club: 1706 (of 2736)
kept at each floor: {300: '85.2%', 600: '77.6%', 900: '69.6%', 1200: '60.7%'} | fee share kept: {300: '93.9%', 600: '89.4%', 900: '84.5%', 1200: '76.5%'}


Work-rate quantities (Sofascore per 90, the 16 shared metrics of Step 5d plus the keeper block)
need the identity join: Sofascore id → Transfermarkt id ← Understat id, then the Understat role.

In [5]:
from scout.data import reep, sofascore, transfermarkt as tm_loader
from scout.identity import build_team_lineage, load_overrides
from scout.panel import identity

comps = list(config.BIG5) + list(config.FEEDERS)
tm_panel = tm_loader.load_player_club_seasons(comps, list(config.SEASONS))
tm_clubs = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
ss = sofascore.load()
us = understat.load("player_season"); us["competition_id"] = us.league.map(LEAGUE_TO_COMP)
lineage = build_team_lineage(tm_clubs, {
    "sofascore": ss[["competition_id", "team_name"]].drop_duplicates(),
    "understat": us[["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"}),
}, load_overrides("teams"))
people = reep.load_people(); tm_side = identity.transfermarkt_side(tm_panel)
ss_ids = identity.resolve_provider("sofascore", ss, tm_side, lineage, people).drop_duplicates("provider_id")[["provider_id", "tm_player_id"]]
us_ids = identity.resolve_provider("understat", us, tm_side, lineage, people).drop_duplicates("provider_id")[["provider_id", "tm_player_id"]]
print("ids:", len(ss_ids), "sofascore |", len(us_ids), "understat")

wr = pd.concat([ss[["competition_id", "season", "sofascore_player_id", "minutesPlayed"]], workrate.sofascore_per90(ss)], axis=1)
wr["tm_player_id"] = wr.sofascore_player_id.astype(int).astype(str).map(ss_ids.set_index("provider_id").tm_player_id)
roles = per90[["competition_id", "season", "player_id", "role", "minutes"]].copy()
roles["tm_player_id"] = roles.player_id.astype(int).astype(str).map(us_ids.set_index("provider_id").tm_player_id)
main_role = roles.sort_values("minutes", ascending=False).drop_duplicates(["competition_id", "season", "tm_player_id"])
wr = wr.dropna(subset=["tm_player_id"]).merge(main_role[["competition_id", "season", "tm_player_id", "role"]], on=["competition_id", "season", "tm_player_id"])
wr["minutes"] = pd.to_numeric(wr.minutesPlayed); wr["player_id"] = wr.tm_player_id
WR = [m for m in workrate.SHARED if m not in ("xg", "xa", "goals", "assists")] + ["possession_won_att_third_sofascore"]
print(len(wr), "Sofascore player-seasons with a Big-5 role")
rows = []
for role in ["GK", "CB", "FB", "CM", "W", "ST"]:
    for floor in [300, 600, 900]:
        r, n = year_to_year(wr[wr.role == role], floor, WR)
        rows.append(pd.concat([pd.Series({"role": role, "floor": floor, "pairs": n}), r.round(2)]))
wr_stability = pd.DataFrame(rows).set_index(["role", "floor"])
print(wr_stability.rename(columns={"possession_won_att_third_sofascore": "poss_won_att3"}).to_string())
print("\nwork-rate quantities with r >= 0.3 at 600, per role:")
for role, group in wr_stability.xs(600, level="floor").iterrows():
    print(f"  {role}: {[q for q in WR if group[q] >= 0.3]}")

ids: 21090 sofascore | 9531 understat
26557 Sofascore player-seasons with a Big-5 role
            pairs  tackles  interceptions  recoveries  clearances  dribbles  key_passes  big_chances_created  accurate_passes  accurate_long_balls  fouls  saves  goals_conceded  poss_won_att3
role floor                                                                                                                                                                                     
GK   300      920     0.07           0.17        0.39        0.45      0.23        0.13                 0.06             0.71                 0.70   0.07   0.30            0.40          -0.01
     600      807     0.07           0.22        0.53        0.51      0.27        0.13                 0.09             0.75                 0.71   0.08   0.36            0.47          -0.01
     900      729     0.08           0.25        0.56        0.49      0.36        0.14                 0.06             0.78                 0.7

/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/mihailandreev/foot

### Step 1 note — quantities per role and the minutes floor, from the two tables above

**Floor: 600 minutes** in a role-season. On the Understat quantities, going from 300 to 600 buys
+0.05 to +0.09 in year-to-year r for every outfield role (ST npxG 0.54 → 0.59, W xA 0.44 → 0.50,
CB shots 0.46 → 0.52); 600 to 900 buys only +0.03 to +0.05 while dropping eight more points of the
backtest population (paid Big-5 departures with a panel row: 77.6% kept at 600 and 89.4% of the
fees, versus 69.6% / 84.5% at 900, 85.2% / 93.9% at 300). Rejected: 300 (the noisiest step and the
largest gain forgone), 900 and 1,200 (less backtest for a smaller gain). Players between 300 and
600 minutes stay in the tables and get profiled from other seasons; they carry no role-season
of their own.

**Quantities that enter, per role** (year-to-year r ≥ 0.3 at 600; goals and assists are never
inputs — the spec builds on expected quantities, and they are also the least stable column in
every role, 0.15–0.45):

| Role | Understat | Work-rate (Sofascore/FotMob shared per 90) |
|---|---|---|
| GK | xGChain, xGBuildup (distribution only, 0.65) | recoveries, clearances, accurate passes, long balls, saves (0.36), goals conceded (0.47 — a team quantity; the keeper's own measure is the Step 6 proxy) |
| CB | npxG (0.30), key passes, shots, xGChain, xGBuildup — not xA (0.24) | tackles, interceptions, recoveries, clearances, dribbles, key passes, accurate passes, long balls, fouls, possession won in the attacking third — not big chances created (0.17) |
| FB, CM, W, ST | npxG, xA, key passes, shots, xGChain, xGBuildup | all twelve, big chances created included (0.36–0.53) |

Two things to carry forward: `goals_conceded` per 90 is stable for outfielders (0.34–0.43) because
it is the team's defence, not the player's — it belongs to plus-minus (Step 2/6), not to a
player's quantity list; and the most stable columns everywhere are volume/style measures
(accurate passes 0.8+, dribbles 0.7+), which say what a player *does*, not how much it is worth —
Step 2 decides what contribution is made of, Step 1 only says which columns carry signal.

Ported: `scout.models.quantities` (`MIN_MINUTES`, `ROLE_QUANTITIES`, `season_role_per90`).